In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.matrix_operations import create_point_matrix
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE, dataset_tp_rp_split

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 10))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)




Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [3]:

tmp = df.sample(1000)

from scripts.beamforming import get_best_beam
from scripts.utils import extract_unique_npcis

tmp['best_beam'] = tmp['measurements_matrix'].apply(lambda x: get_best_beam(x, rf_param))

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 42)

unique_pcis = extract_unique_npcis(tmp['measurements_matrix'])

m_rp, idx_rp = create_point_matrix(df_rp, unique_pcis, rf_param)

tp = df_tp.iloc[2]

tp


lat                                                            41.898334
lng                                                            12.429553
measurements_matrix         pci  beam_index  nr_arfcn  operator_id   ...
campaign_id                                                            1
Name: 2, dtype: object